In [44]:
print("Daniel Sebastian Macias ISC 745455")

Daniel Sebastian Macias ISC 745455


In [45]:
import findspark
findspark.init()

from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, FloatType
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("QuickCommerceAnalysis").getOrCreate()

# Define corrected schema
QuickCommerce_schema = StructType([
    StructField("Order_ID", LongType(), True),
    StructField("Company", StringType(), True),
    StructField("City", StringType(), True),
    StructField("CustomerAge", IntegerType(), True),
    StructField("Order_value", FloatType(), True),
    StructField("Delivery_time_min", FloatType(), True),
    StructField("Distance_km", FloatType(), True),
    StructField("Items_count", FloatType(), True),
    StructField("Product_category", StringType(), True),
    StructField("Payment_method", StringType(), True),
    StructField("Customer-rating", FloatType(), True),
    StructField("Discount_applied", FloatType(), True),
    StructField("Delivery_partner_rating", FloatType(), True)
])

# Load data
csv_path = r"/opt/spark/work-dir/data/quick_commerce_data_raw.csv"
df = spark.read.csv(csv_path, header=True, schema=QuickCommerce_schema)

# Show schema and data
df.printSchema()
df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- CustomerAge: integer (nullable = true)
 |-- Order_value: float (nullable = true)
 |-- Delivery_time_min: float (nullable = true)
 |-- Distance_km: float (nullable = true)
 |-- Items_count: float (nullable = true)
 |-- Product_category: string (nullable = true)
 |-- Payment_method: string (nullable = true)
 |-- Customer-rating: float (nullable = true)
 |-- Discount_applied: float (nullable = true)
 |-- Delivery_partner_rating: float (nullable = true)

+--------+----------------+--------+-----------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|CustomerAge|Order_value|Delivery_time_min|Distance_km|Items_count|Product_category|  Payment_method|Customer-rating|Discount_applied|Delivery_partner_rating|
+--------+----------------+-------

26/02/24 01:58:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Order_ID, Company, City, Customer_Age, Order_Value, Delivery_Time_Min, Distance_Km, Items_Count, Product_Category, Payment_Method, Customer_Rating, Discount_Applied, Delivery_Partner_Rating
 Schema: Order_ID, Company, City, CustomerAge, Order_value, Delivery_time_min, Distance_km, Items_count, Product_category, Payment_method, Customer-rating, Discount_applied, Delivery_partner_rating
Expected: CustomerAge but found: Customer_Age
CSV file: file:///opt/spark/work-dir/data/quick_commerce_data_raw.csv


In [46]:
import findspark
findspark.init()

from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, FloatType
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("QuickCommerceAnalysis").getOrCreate()

# Schema MUST match CSV header exactly
QuickCommerce_schema = StructType([
    StructField("Order_ID", LongType(), True),
    StructField("Company", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Customer_Age", IntegerType(), True),
    StructField("Order_Value", FloatType(), True),
    StructField("Delivery_Time_Min", FloatType(), True),
    StructField("Distance_Km", FloatType(), True),
    StructField("Items_Count", FloatType(), True),
    StructField("Product_Category", StringType(), True),
    StructField("Payment_Method", StringType(), True),
    StructField("Customer_Rating", FloatType(), True),
    StructField("Discount_Applied", FloatType(), True),
    StructField("Delivery_Partner_Rating", FloatType(), True)
])

# Load data
csv_path = "/opt/spark/work-dir/data/quick_commerce_data_raw.csv"

df = (
    spark.read
    .option("header", "true")
    .schema(QuickCommerce_schema)
    .csv(csv_path)
)

# Show schema and data
df.printSchema()
df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [47]:
qcommerce_wo_nulls = df.dropna()
print(f"size of data frame after removing nulls: {qcommerce_wo_nulls.count()}")

[Stage 21:=======>                                                  (1 + 7) / 8]

size of data frame after removing nulls: 780992


In [48]:
qcommerce_wo_nulls_fillna = df.fillna({
'City': 'Uknown',
'Items_Count': 0.0,
'Customer_Rating': 0.0,
'Delivery_Partner_Rating': 0.0

})

print(f"size of data frame after removing nulls with 'fillna': {qcommerce_wo_nulls_fillna.count()}")

size of data frame after removing nulls with 'fillna': 1000000


In [49]:
from pyspark.sql. functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [50]:
from pyspark. sql. functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

In [51]:
from pyspark.sql.functions import when, col

qcommerce_df_task1 = df.withColumn(
    "Delivery_SLA",
    when(col("Delivery_Time_Min") <= 15, "FAST")
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME")
    .otherwise("LATE")
)

qcommerce_df_task1.select(
    "Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA"
).show()


+--------+----------------+---------+-----------------+------------+
|Order_ID|         Company|     City|Delivery_Time_Min|Delivery_SLA|
+--------+----------------+---------+-----------------+------------+
| 1000001|Swiggy Instamart|    Noida|           19.182|     ON-TIME|
| 1000002|Flipkart Minutes| Amritsar|           19.644|     ON-TIME|
| 1000003|Flipkart Minutes|   Mumbai|            16.91|     ON-TIME|
| 1000004|Swiggy Instamart|    Delhi|            5.864|        FAST|
| 1000005|           Dunzo|   Mumbai|            12.47|        FAST|
| 1000006|        Jio Mart|  Kolkata|           19.738|     ON-TIME|
| 1000007|         Blinkit| Bengluru|           18.476|     ON-TIME|
| 1000008|           Dunzo|  Chennai|           10.916|        FAST|
| 1000009|Flipkart Minutes| Bengluru|           18.174|     ON-TIME|
| 1000010|Swiggy Instamart|  Chennai|           11.238|        FAST|
| 1000011|         Blinkit|Hyderabad|           12.838|        FAST|
| 1000012|Swiggy Instamart|     Pu

In [52]:
from pyspark.sql.functions import avg, count

qcommerce_df_task2 = qcommerce_df_task1.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))) \
    .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
        .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
        .when(col("Effective_Order_Value") > 500, "HIGH")) \
    .groupBy("Price_Bucket").agg(
        count("*").alias("Count_Price_Bucket"),
        avg("Effective_Order_Value").alias("Avg_Effective_Value")
    ) \
    .orderBy("Avg_Effective_Value", ascending=False)

qcommerce_df_task2.show()

+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Value|
+------------+------------------+-------------------+
|        HIGH|            268953|  740.4337238893675|
|      MEDIUM|            210429|  358.0973233400432|
|         LOW|            520618| 21.591204157544553|
+------------+------------------+-------------------+



In [53]:
from pyspark.sql.functions import when, col, count, avg

qcommerce_df_task3 = df.filter(col("Customer_Age").isNotNull() & (col("Customer_Age") >= 0) & (col("Customer_Age") <= 100)) \
                                       .withColumn("Age_Group",when(col("Customer_Age")< 25, "YOUNG")
                                                              .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT")
                                                              .otherwise("SENIOR")) \
                                       .groupBy("Age_Group", "Product_Category").agg(count("*").alias("orders"),avg("Order_Value").alias("avg_order_value")) \
                                       .orderBy("Age_Group", ascending=True) \
                                       .orderBy(col("Age_Group").asc(), col("orders").desc())

qcommerce_df_task3.show()

[Stage 33:=======>                                                  (1 + 7) / 8]

+---------+-------------------+------+-----------------+
|Age_Group|   Product_Category|orders|  avg_order_value|
+---------+-------------------+------+-----------------+
|    ADULT|              Dairy| 68512|573.4268899414931|
|    ADULT|      Personal Care| 68331| 569.512671998805|
|    ADULT|          Groceries| 68291|571.5250993844182|
|    ADULT|          Household| 68110|572.9259149188181|
|    ADULT|             Snacks| 68100|570.3797974095505|
|    ADULT|          Beverages| 67979|572.0329877095578|
|    ADULT|Fruits & Vegetables| 67736|569.8593599596651|
|   SENIOR|          Groceries| 51030|572.2596391052539|
|   SENIOR|              Dairy| 51025| 573.781117268945|
|   SENIOR|             Snacks| 50924| 572.687851608881|
|   SENIOR|          Household| 50803| 571.172082385334|
|   SENIOR|          Beverages| 50746| 568.141013231256|
|   SENIOR|      Personal Care| 50707|571.1642560109325|
|   SENIOR|Fruits & Vegetables| 50642|573.7244422534909|
|    YOUNG|              Dairy|